# Extract Tables from ETrade_Morningstar ETF Report Card .pdf into .json

Extracts .pdf tables into .json suitable for [import into DynamoDB](../backend/dyndb_load_json/dyndb_load_json.py) having structure based upon example output file [src/IDMO.msReportCard.json](IDMO.msReportCard.json).

(Tried extracting .json from the MinerU markdown, but results are inconsistent (skipped data), so using this separate PDF process.)

## Pipeline:
1.  manually download E*Trade ETF Report Cards PDF files to /data/raw/downloads
2.  run mineru.ipynb to convert PDF to Markdown stored in /data/processed/{file_type}/{yyyy-mm-dd}/*.pdf.md
    - filetype == 'ETrade_Morningstar' for the report cards (among other file_types downloaded from ETrade & elsewhere)
    - md is prefixed with Obsidian properties
3.  run etrade_morningstar_extract.ipynb to convert PDF to JSON (including Obsidian properties created in step 2)
    - output stored in /data/processed/{file_type}/{yyyy-mm-dd}/*.json
4.  JSON files uploaded to S3 initiate Lambda to store in DynamoDB (triggered by ObjectCreated)

## ToDo: 
- dynamodb_client.py -- enhance top_10_holdings to include symbol and cik(opt), along with name & weight_pct.  maybe also sector & industry
- db_query.py -- enhance get_top_10_holdings retieve additional 'real-time' info
- fastinfo["quoteType"] = ETF|EQUITY




In [13]:
# initialize
import os
import glob
from pathlib import Path
from dotenv import load_dotenv
import json

load_dotenv(override=True)

from datetime import datetime
today_str = datetime.today().strftime("%Y-%m-%d")
#today_str = "2026-11-01"

WIKI_FOLDER="/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/wiki-proto"
md_folder = f"{WIKI_FOLDER}/data/processed/ETrade_Morningstar/{today_str}"
os.environ["MARKDOWN_FOLDER"] = md_folder

json_folder = f"{md_folder}/json"
os.environ["JSON_FOLDER"] = json_folder

os.makedirs(json_folder, exist_ok=True)

print(f"md_folder={md_folder}")
print(f"json_folder={json_folder}")

md_filepaths = glob.glob(md_folder + "/*.md")
md_filepaths

md_folder=/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/wiki-proto/data/processed/ETrade_Morningstar/2026-11-08
json_folder=/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/wiki-proto/data/processed/ETrade_Morningstar/2026-11-08/json


['/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/wiki-proto/data/processed/ETrade_Morningstar/2026-11-08/iShares_Silver_Trust__Exchange_Traded_Fund_Report_Card.pdf.md',
 '/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/wiki-proto/data/processed/ETrade_Morningstar/2026-11-08/SPDR_Gold_Shares__Exchange_Traded_Fund_Report_Card.pdf.md',
 '/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/wiki-proto/data/processed/ETrade_Morningstar/2026-11-08/VanEck_Bitcoin_ETF__Exchange_Traded_Fund_Report_Card.pdf.md']

In [14]:
def get_properties_from_markdown(md = None, md_filepath = None) -> dict:
    """Extract properties from markdown text if available."""
    if md is None and md_filepath is not None:
        with open(md_filepath, "r") as md_file:
            md = md_file.read()

    # extract Obsidian properties from the markdown front matter
    properties_str = md.split("---\n")[1]
    properties = {}
    for line in properties_str.split("\n"):
        if ":" in line:
            key, value = line.split(":", 1)
            properties[key.strip()] = value.strip()
    return properties

#date_str = "2026-11-01"
#md_folder = f"{WIKI_FOLDER}/data/processed/ETrade_Morningstar/{date_str}"
print(md_folder)
md_filepaths = glob.glob(md_folder + "/*.md")
for md_file in md_filepaths:
    print(md_file)
    properties = get_properties_from_markdown(md_filepath=md_file)
    print(properties)
    #break


/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/wiki-proto/data/processed/ETrade_Morningstar/2026-11-08
/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/wiki-proto/data/processed/ETrade_Morningstar/2026-11-08/iShares_Silver_Trust__Exchange_Traded_Fund_Report_Card.pdf.md
{'title': 'iShares Silver Trust — Exchange Traded Fund Report Card', 'source': 'ETrade_Morningstar', 'topic': 'Exchange-traded fund report; silver', 'author': 'n/a', 'published': '2026-11-07', 'company_name': 'iShares Silver Trust', 'symbol': 'SLV', 'asof_date': '2026-11-07', 'source_file': '"[[data/raw/downloads/z4f907d0azcd7aca12270d41c5898c6f25fb4a3094.pdf]]"', 'created': '2026-11-08', 'description': 'ETF report card for iShares Silver Trust (SLV), which seeks to reflect the performance of silver prices before expenses. It summarizes fund characteristics, historical returns, portfolio facts, and risk statistics. Report created November 7, 2026, with data dated through October 31–November 6, 2026.'}
/mnt/c/Us

In [15]:
## Bedrock / LiteLLM configuration
REGION = os.environ.get("BEDROCK_REGION", "us-east-1")
os.environ["AWS_REGION_NAME"] = REGION  # LiteLLM's preferred variable
os.environ["AWS_REGION"] = REGION  # Boto3 standard
os.environ["AWS_DEFAULT_REGION"] = REGION  # Fallback

BEDROCK_MODEL_ID = os.environ.get("BEDROCK_MODEL_ID", "us.amazon.nova-pro-v1:0")
MODEL = f"bedrock/{BEDROCK_MODEL_ID}"

schema_folder = os.getenv("SCHEMA_FOLDER", "./")
SCHEMA_EXAMPLE_PATH = f"{schema_folder}/IDMO.msReportCard.json"

pdf_folder = os.getenv("PDF_FOLDER", "./")
md_folder = os.getenv("MARKDOWN_FOLDER", "./")

INSTRUCTIONS = """You convert E*TRADE Exchange Traded Fund Report Card text into structured JSON.

You will be given the raw text extracted from a *.msReportCard.pdf file. Produce a JSON object
that follows EXACTLY the same keys, nesting, and structure as this example (the example's values
belong to a different fund and must not be reused):

{example_json}

Rules:
- Use the ticker symbol found in the source text for "symbol".
- "asof_dt" is the "Average Annual Return & Tax Analysis as of" date in MM/DD/YY format
  (this is usually a month-end date, e.g. 07/31/26), NOT the "Report created on" date and NOT the
  "Close on" date.
- Morningstar star ratings (e.g. "morningstar_star_rating" and "rating" fields) must use the
  literal capital letter "H" repeated once per star (e.g. 5 stars = "HHHHH"), not the "\u2605" star
  glyph.
- Numbers must be plain JSON numbers (no "%", "$", or "," characters); percentages are represented
  as their numeric value (e.g. 23.91 for 23.91%).
- If a value is not present in the source text, use null.
- Respond with ONLY the JSON object, no markdown fences, no commentary.
"""

from pypdf import PdfReader

def extract_pdf_text(pdf_path: Path) -> str:
    """Extract concatenated text from every page of a PDF."""
    reader = PdfReader(str(pdf_path))
    return "\n".join(page.extract_text() or "" for page in reader.pages)

def _parse_json_response(text: str) -> dict:
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`")
        cleaned = cleaned.split("\n", 1)[1] if "\n" in cleaned else cleaned
    start, end = cleaned.find("{"), cleaned.rfind("}")
    if start == -1 or end == -1:
        raise ValueError(f"No JSON object found in model response: {text[:200]!r}")
    return json.loads(cleaned[start : end + 1])

async def convert_pdf_text_to_json(pdf_text: str) -> dict:
    """Send extracted PDF text to Bedrock (via LitellmModel) and return parsed JSON."""
    from agents import Agent, ModelSettings, Runner, trace
    from agents.extensions.models.litellm_model import LitellmModel

    with open(SCHEMA_EXAMPLE_PATH) as f:
        example_json = f.read()

    agent = Agent(
        name="msReportCard Converter",
        instructions=INSTRUCTIONS.format(example_json=example_json),
        model=LitellmModel(model=MODEL),
        model_settings=ModelSettings(max_tokens=4096),
    )
    with trace("msrportcard_converter"):
        result = await Runner.run(agent, input=pdf_text)
    
    return _parse_json_response(result.final_output)


In [16]:
# process *.pdf files and upload corresponding JSON to S3
import boto3

#date_str = "2026-11-01"
#md_folder = f"{WIKI_FOLDER}/data/processed/ETrade_Morningstar/{date_str}"
print(md_folder)
print(json_folder)
md_filepaths = glob.glob(md_folder + "/*.md")
#md_filepaths = md_filepaths[6:7] # XLK
#md_filepaths

json_folder = os.getenv("JSON_FOLDER", f"{md_folder}/json")

for md_filepath in md_filepaths:
    print(md_filepath)
    properties = get_properties_from_markdown(md_filepath=md_filepath)
    md_symbol = properties["symbol"]
    print(properties)

    # get .pdf path from .md properties
    source_file = properties.get("source_file").replace('"[[', '').replace(']]"', '')
    pdf_path = f"{WIKI_FOLDER}/{source_file}"
    print(pdf_path)

    pdf_text = extract_pdf_text(pdf_path)
    json_data = await convert_pdf_text_to_json(pdf_text)
    json_data["symbol"] = md_symbol
    json_data["properties"] = properties

    json_path = f"{json_folder}/{md_symbol}.msReportCard.json"
    with open(json_path, "w") as f:
        json.dump(json_data, f, indent=4)
    print(f"Wrote {json_path}")

    # upload .json to S3 to be processed by ObjectCreated-initiated Lambda ramlytics_dyndb_load_json
    s3_bucket = os.getenv("S3_BUCKET", "fh-danelfin-289755104220")
    s3_inbound_folder = "inbound/market_data"
    s3_client = boto3.client("s3")
    s3_key = f"{s3_inbound_folder}/{md_symbol}.msReportCard.json"
    s3_client.upload_file(str(json_path), s3_bucket, s3_key)
    print(f"Uploaded to s3://{s3_bucket}/{s3_key}")


/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/wiki-proto/data/processed/ETrade_Morningstar/2026-11-08
/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/wiki-proto/data/processed/ETrade_Morningstar/2026-11-08/json
/mnt/c/Users/richa/OneDrive/Documents/ramlytics_data/wiki-proto/data/processed/ETrade_Morningstar/2026-11-08/iShares_Silver_Trust__Exchange_Traded_Fund_Report_Card.pdf.md
{'title': 'iShares Silver Trust — Exchange Traded Fund Report Card', 'source': 'ETrade_Morningstar', 'topic': 'Exchange-traded fund report; silver', 'author': 'n/a', 'published': '2026-11-07', 'company_name': 'iShares Silver Trust', 'symbol': 'SLV', 'asof_date': '2026-11-07', 'source_file': '"[[data/raw/downloads/z4f907d0azcd7aca12270d41c5898c6f25fb4a3094.pdf]]"', 'created': '2026-11-08', 'description': 'ETF report card for iShares Silver Trust (SLV), which seeks to reflect the performance of silver prices before expenses. It summarizes fund characteristics, historical returns, portfolio facts, and 